## This Notebook for Testing the library

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

sys.path.insert(0, str(project_root))

In [2]:

# ============================================
# ZATML vs SCIKIT-LEARN
# Logistic Regression Benchmark
# ============================================

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression as SKLogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Your library
from ZatML.Models.logisticRegression import LogisticRegression as ZatLogisticRegression
from ZatML.Utils.metric import Metrices


# ============================================
# CONFIGURATION
# ============================================

RANDOM_STATE = 42
TEST_SIZE = 0.2

ZAT_ITERATIONS = 2000
ZAT_LEARNING_RATE = 0.1

N_SPLITS = 5


# ============================================
# 1. LOAD DATASET
# ============================================

dataset = load_breast_cancer()

X = dataset.data

# 0 = Benign
# 1 = Malignant
y = (dataset.target == 0).astype(np.int64)

print("=" * 55)
print("DATASET INFORMATION")
print("=" * 55)

print("Dataset:", "Breast Cancer Wisconsin")
print("Samples:", X.shape[0])
print("Features:", X.shape[1])
print("Benign:", np.sum(y == 0))
print("Malignant:", np.sum(y == 1))


# ============================================
# 2. METRICS FUNCTION
# ============================================

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1 Score": f1_score(
            y_true, y_pred, zero_division=0
        )
    }


def print_confusion_matrix(y_true, y_pred, model_name):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    print(f"\n{model_name} Confusion Matrix")
    print("                 Predicted")
    print("               Benign  Malignant")
    print(f"Actual Benign    {cm[0,0]:4d}    {cm[0,1]:4d}")
    print(f"Actual Malignant {cm[1,0]:4d}    {cm[1,1]:4d}")


def print_metrics(metrics, model_name):

    print(f"\n{model_name}")
    print("-" * 40)

    for metric, value in metrics.items():
        print(f"{metric:<15}: {value:.4f}")


# ============================================
# 3. TRAIN / TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)


# ============================================
# 4. FEATURE SCALING
# ============================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train_2d = y_train.reshape(-1, 1)


# ============================================
# 5. TRAIN ZATML
# ============================================

print("\n" + "=" * 55)
print("TRAINING ZATML")
print("=" * 55)

np.random.seed(RANDOM_STATE)

zat_model = ZatLogisticRegression(
    num_iter=ZAT_ITERATIONS,
    show_logs=False,
    lr=ZAT_LEARNING_RATE
)

zat_model.fit(X_train, y_train_2d)

zat_probabilities, zat_predictions = zat_model.predict(X_test)

zat_predictions = np.asarray(zat_predictions).reshape(-1)
zat_probabilities = np.asarray(zat_probabilities).reshape(-1)


# ============================================
# 6. TRAIN SCIKIT-LEARN
# ============================================

print("\n" + "=" * 55)
print("TRAINING SCIKIT-LEARN")
print("=" * 55)

sk_model = SKLogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=5000,
    tol=1e-10
)

sk_model.fit(X_train, y_train)

sk_predictions = sk_model.predict(X_test)

sk_probabilities = sk_model.predict_proba(X_test)[:, 1]


print("Scikit-learn iterations:", sk_model.n_iter_)


# ============================================
# 7. EVALUATE BOTH MODELS
# ============================================

zat_results = calculate_metrics(
    y_test,
    zat_predictions
)

sk_results = calculate_metrics(
    y_test,
    sk_predictions
)

print("\n" + "=" * 55)
print("FINAL TEST RESULTS")
print("=" * 55)

comparison = pd.DataFrame(
    [zat_results, sk_results],
    index=["ZATML", "Scikit-learn"]
)

print(comparison.round(4))


# ============================================
# 8. CONFUSION MATRICES
# ============================================

print_confusion_matrix(
    y_test,
    zat_predictions,
    "ZATML"
)

print_confusion_matrix(
    y_test,
    sk_predictions,
    "SCIKIT-LEARN"
)


# ============================================
# 9. CLASSIFICATION REPORT
# ============================================

print("\n" + "=" * 55)
print("ZATML CLASSIFICATION REPORT")
print("=" * 55)

print(
    classification_report(
        y_test,
        zat_predictions,
        labels=[0, 1],
        target_names=["Benign", "Malignant"],
        zero_division=0
    )
)

print("\n" + "=" * 55)
print("SCIKIT-LEARN CLASSIFICATION REPORT")
print("=" * 55)

print(
    classification_report(
        y_test,
        sk_predictions,
        labels=[0, 1],
        target_names=["Benign", "Malignant"],
        zero_division=0
    )
)


# ============================================
# 10. PREDICTION AGREEMENT
# ============================================

agreement = np.mean(
    zat_predictions == sk_predictions
)

disagreement = np.sum(
    zat_predictions != sk_predictions
)

print("\n" + "=" * 55)
print("PREDICTION COMPARISON")
print("=" * 55)

print(f"Agreement    : {agreement:.4f}")
print(f"Disagreement : {disagreement}")
print(f"Test Samples : {len(y_test)}")


# ============================================
# 11. ZATML METRICS IMPLEMENTATION TEST
# ============================================

print("\n" + "=" * 55)
print("TESTING YOUR ZATML METRICS")
print("=" * 55)

def evaluate_zatml_metrics(y_true, y_pred):

    # Create a new instance for each evaluation
    metric = Metrices()

    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    return {
        "Accuracy": metric.accuracy(y_pred, y_true),
        "Precision": metric.precesion(y_pred, y_true),
        "Recall": metric.recall(y_pred, y_true),
        "F1 Score": metric.f1_score(y_pred, y_true)
    }


for name, predictions in [
    ("ZATML predictions", zat_predictions),
    ("Scikit-learn predictions", sk_predictions)
]:

    print_metrics(
        evaluate_zatml_metrics(y_test, predictions),
        name + " - ZATML Metrics"
    )

    print_metrics(
        calculate_metrics(y_test, predictions),
        name + " - Reference Metrics"
    )


# ============================================
# 12. 5-FOLD CROSS VALIDATION
# ============================================

print("\n" + "=" * 55)
print("5-FOLD CROSS VALIDATION")
print("=" * 55)

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_results = []

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X, y), start=1
):

    print(f"\nRunning Fold {fold}/{N_SPLITS}")

    # Split raw data
    X_tr = X[train_idx]
    X_val = X[valid_idx]

    y_tr = y[train_idx]
    y_val = y[valid_idx]

    # Fit scaler only on the training fold
    fold_scaler = StandardScaler()

    X_tr = fold_scaler.fit_transform(X_tr)
    X_val = fold_scaler.transform(X_val)

    # ----------------------------------------
    # ZATML
    # ----------------------------------------

    np.random.seed(RANDOM_STATE + fold)

    fold_zat_model = ZatLogisticRegression(
        num_iter=ZAT_ITERATIONS,
        show_logs=False,
        lr=ZAT_LEARNING_RATE
    )

    fold_zat_model.fit(
        X_tr,
        y_tr.reshape(-1, 1)
    )

    _, fold_zat_predictions = fold_zat_model.predict(X_val)

    fold_zat_predictions = np.asarray(
        fold_zat_predictions
    ).reshape(-1)

    zat_fold_metrics = calculate_metrics(
        y_val,
        fold_zat_predictions
    )

    cv_results.append({
        "Fold": fold,
        "Model": "ZATML",
        **zat_fold_metrics
    })

    # ----------------------------------------
    # SCIKIT-LEARN
    # ----------------------------------------

    fold_sk_model = SKLogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=5000,
        tol=1e-10
    )

    fold_sk_model.fit(X_tr, y_tr)

    fold_sk_predictions = fold_sk_model.predict(X_val)

    sk_fold_metrics = calculate_metrics(
        y_val,
        fold_sk_predictions
    )

    cv_results.append({
        "Fold": fold,
        "Model": "Scikit-learn",
        **sk_fold_metrics
    })


# ============================================
# 13. CROSS VALIDATION RESULTS
# ============================================

cv_df = pd.DataFrame(cv_results)

print("\nIndividual Fold Results:")
print(cv_df.round(4).to_string(index=False))

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
]

cv_summary = cv_df.groupby("Model")[metric_columns].agg(
    ["mean", "std"]
)

print("\n" + "=" * 55)
print("CROSS VALIDATION SUMMARY")
print("=" * 55)

print(cv_summary.round(4))


# ============================================
# 14. FINAL INTERPRETATION
# ============================================

print("\n" + "=" * 55)
print("BENCHMARK COMPLETED")
print("=" * 55)

print("Holdout comparison:")
print(comparison.round(4))

print("\nCross-validation mean F1 scores:")

print(
    cv_summary["F1 Score"]["mean"].sort_values(
        ascending=False
    )
)


DATASET INFORMATION
Dataset: Breast Cancer Wisconsin
Samples: 569
Features: 30
Benign: 357
Malignant: 212

TRAINING ZATML

TRAINING SCIKIT-LEARN
Scikit-learn iterations: [57]

FINAL TEST RESULTS
              Accuracy  Precision  Recall  F1 Score
ZATML           0.9737     0.9756  0.9524    0.9639
Scikit-learn    0.9386     0.9487  0.8810    0.9136

ZATML Confusion Matrix
                 Predicted
               Benign  Malignant
Actual Benign      71       1
Actual Malignant    2      40

SCIKIT-LEARN Confusion Matrix
                 Predicted
               Benign  Malignant
Actual Benign      70       2
Actual Malignant    5      37

ZATML CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Benign       0.97      0.99      0.98        72
   Malignant       0.98      0.95      0.96        42

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114